# GlucoTwin: 01. Exploratory Data Analysis
**Team:** Aethan, Amity University Mumbai  
**Challenge:** Digital Twin Challenge 2026 — Happiest Health Summit  

This notebook explores the raw multi-modal datasets:
1. Static Electronic Health Records (EHR) of 200 Indian Type 2 Diabetes patients.
2. Dynamic continuous wearable and CGM telemetry (1.73M 5-minute readings).
3. Postprandial glucose dynamics, dawn phenomenon, and Indian dietary patterns.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Project root:", PROJECT_ROOT)



## 1. Load & Inspect Static EHR Cohort


In [ ]:
ehr_path = PROJECT_ROOT / "data" / "raw" / "ehr_patients.csv"
ehr_df = pd.read_csv(ehr_path)
print(f"Total Enrolled Patients: {len(ehr_df)}")
ehr_df.head()



In [ ]:
# Summary statistics for clinical biomarkers
biomarkers = ['age', 'bmi', 'hba1c', 'fasting_glucose', 'blood_pressure_systolic', 'cholesterol_total']
ehr_df[biomarkers].describe().T



In [ ]:
# Distribution of HbA1c and Asian Indian BMI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(ehr_df['hba1c'], kde=True, ax=axes[0], color='#0284c7', bins=20)
axes[0].axvline(7.0, color='green', linestyle='--', label='Target Goal (<7.0%)')
axes[0].axvline(9.0, color='red', linestyle='--', label='High Risk (>9.0%)')
axes[0].set_title("HbA1c (%) Distribution in Indian T2D Cohort", fontsize=12, fontweight='bold')
axes[0].legend()

sns.histplot(ehr_df['bmi'], kde=True, ax=axes[1], color='#0d9488', bins=20)
axes[1].axvline(23.0, color='orange', linestyle='--', label='Asian Indian Overweight (>23.0)')
axes[1].axvline(25.0, color='red', linestyle='--', label='Asian Indian Obese (>25.0)')
axes[1].set_title("Body Mass Index (BMI) Distribution (ICMR Guidelines)", fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()



## 2. Load & Inspect Dynamic CGM Telemetry


In [ ]:
# Read first 10,000 readings of continuous CGM telemetry
cgm_path = PROJECT_ROOT / "data" / "raw" / "cgm_readings.csv"
cgm_sample = pd.read_csv(cgm_path, nrows=10000)
cgm_sample['timestamp'] = pd.to_datetime(cgm_sample['timestamp'])
print("CGM Telemetry Columns:", list(cgm_sample.columns))
cgm_sample.head()



In [ ]:
# Plot 48-Hour Interstitial Glucose Trajectory for Patient P001
p1_readings = cgm_sample[cgm_sample['patient_id'] == 'P001'].head(288 * 2) # 48 hours

plt.figure(figsize=(15, 6))
plt.plot(p1_readings['timestamp'], p1_readings['glucose_mg_dl'], color='#0284c7', lw=1.8, label='CGM Interstitial Glucose')
plt.axhspan(70, 180, color='green', alpha=0.15, label='Target Euglycemic Zone (70-180 mg/dL)')
plt.axhline(180, color='orange', linestyle='--', alpha=0.7, label='Spike Threshold (180 mg/dL)')
plt.axhline(250, color='red', linestyle='--', alpha=0.7, label='Critical Hyperglycemia (>250 mg/dL)')

plt.title("48-Hour Continuous Glucose Telemetry — Patient P001 (Mumbai, 52y, HbA1c 8.7%)", fontsize=14, fontweight='bold')
plt.xlabel("Timeline", fontsize=11)
plt.ylabel("Glucose (mg/dL)", fontsize=11)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

